## 분석 목적
춘천시 아동급식카드 가맹점 목록과 춘천시 요식업소 현황을 **상호명 및 주소**로 비교해 공식 업태 정보를 연결한다. 이 결과를 바탕으로 주점 관련 업태가 기재된 가맹점을 **추가 점검 후보**로 추출한다.

## 작업 범위와 해석 주의
- 처음 수집한 원본 CSV는 6종이지만, **이번 매칭 코드에서 직접 사용하는 파일은 2종**이다. 나머지 4종은 이후 학교 위치·상가 정보·가격 분석에 활용할 수 있다.
- 완전히 동일한 행을 제거한 뒤 **가맹점 1행을 분석 단위**로 사용한다. 반드시 실제 점포 1곳과 일치하는 것은 아니다.
- `1차매칭`은 **이름과 정규화된 전체 주소가 일치하는 자동 매칭**을 뜻한다. 사람이 실제 영업 여부를 검증했다는 의미는 아니다.
- `검증대기`는 상호명과 도로명 건물번호 또는 지번이 일치한 **추가 후보**다. 실제 동일 업소 여부는 수동 검증이 필요하다.
- `호프/통닭`에는 일반 치킨집도 포함된다. **주점 업태 후보 = 부적합 업소**로 해석하면 안 된다.

**실행 방법:** 아래 코드 셀을 위에서 아래로 순서대로 실행한다. 원본 데이터는 Google Drive의 `MyDrive/data`에 저장되어 있어야 한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. 데이터 불러오기

Google Drive를 연결하고 원본 CSV 2종을 읽는다. 한글 파일명이 자모 분리(NFD) 방식으로 표시되는 경우에도 읽을 수 있도록 처리했다.

**입력 데이터:**
- `강원특별자치도 춘천시_아동급식카드 가맹점_20260827.csv`
- `강원특별자치도 춘천시_요식업소 현황_20260527.csv`

In [ ]:
import os
import re
import unicodedata
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/data"
OUTPUT_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 파일 이름이 자모 분리 형태여도 찾을 수 있도록 통일해서 비교
def find_csv(filename):
    target = unicodedata.normalize("NFC", filename)
    for name in os.listdir(DATA_DIR):
        if unicodedata.normalize("NFC", name) == target:
            return os.path.join(DATA_DIR, name)
    raise FileNotFoundError(f"data 폴더에서 파일을 찾을 수 없습니다: {filename}")


def read_csv_file(filename):
    path = find_csv(filename)
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


card = read_csv_file("강원특별자치도 춘천시_아동급식카드 가맹점_20260827.csv")
restaurant = read_csv_file("강원특별자치도 춘천시_요식업소 현황_20260527.csv")

# 원본 가맹점 파일에 있는 컬럼명 오타 수정
card = card.rename(columns={"기맹점명": "가맹점명"})

print("급식카드 가맹점 원본:", card.shape)
print("요식업소 현황 원본:", restaurant.shape)

Mounted at /content/drive
급식카드 가맹점 원본: (4593, 10)
요식업소 현황 원본: (6184, 7)


## 2. 결측치 및 중복 처리

각 데이터의 결측치 개수를 확인하고 **모든 컬럼 값이 완전히 동일한 행만** 제거한다. 이름이나 주소가 같더라도 업종 등 다른 값이 있으면 삭제하지 않는다.

In [ ]:
print("[원본 결측치 점검]")
for title, data in [("가맹점", card), ("요식업소", restaurant)]:
    print(f"{title}: 전체 {len(data):,}행 / 결측값 {data.isna().sum().sum():,}개")

card_clean = card.drop_duplicates().copy()
restaurant_clean = restaurant.drop_duplicates().copy()

print("\n[완전 중복 제거 결과]")
print(f"가맹점: {len(card):,} → {len(card_clean):,}행")
print(f"요식업소: {len(restaurant):,} → {len(restaurant_clean):,}행")

# 삭제 이전 행 번호를 고유 ID로 사용: 이후 결합에서 중복 방지
card_data = card_clean.reset_index().rename(columns={"index": "card_id"})
restaurant_data = restaurant_clean.reset_index().rename(columns={"index": "restaurant_id"})

assert card_data["card_id"].is_unique
assert restaurant_data["restaurant_id"].is_unique

[원본 결측치 점검]
가맹점: 전체 4,593행 / 결측값 0개
요식업소: 전체 6,184행 / 결측값 2,285개

[완전 중복 제거 결과]
가맹점: 4,593 → 4,568행
요식업소: 6,184 → 6,183행


## 3. 상호명과 주소 정규화

공백을 제거해 상호명을 비교하고, `강원`, `강원도`, `강원특별자치도`의 표기를 통일한다. 이후 **도로명+건물번호** 또는 **동·리+지번**을 추출한다.

주소의 쉼표나 상세주소를 무작정 삭제하면 `1,2동` 등의 의미가 달라질 수 있으므로 원래 정보를 유지한다. 정규화 결과는 자동 매칭을 위한 비교용이며 주소 진위 확인을 대신하지 않는다.

In [ ]:
def clean_name(name):
    if pd.isna(name):
        return ""
    return re.sub(r"\s+", "", str(name).strip())


def clean_address(address):
    if pd.isna(address):
        return ""
    address = str(address).strip().replace("강원특별자치도", "강원도")
    address = re.sub(r"^강원(?=\s+춘천시)", "강원도", address)
    return re.sub(r"\s+", " ", address).strip()


def get_road_address(address):
    if pd.isna(address):
        return ""
    pattern = r"([가-힣A-Za-z0-9·.-]+(?:대로|로|길))\s*(\d+(?:-\d+)?)\b"
    match = re.search(pattern, str(address))
    return match.group(1) + " " + match.group(2) if match else ""


def get_jibun(address):
    if pd.isna(address):
        return ""
    pattern = r"([가-힣0-9]+(?:동|리))\s*(산\s*)?(\d+)(?:\s*-\s*(\d+))?\s*(?:번지)?"
    match = re.search(pattern, str(address).strip())
    if not match:
        return ""
    dong = match.group(1)
    mountain = "산" if match.group(2) else ""
    number = match.group(3)
    if match.group(4):
        number += "-" + match.group(4)
    return dong + " " + mountain + number


card_data["이름_정규화"] = card_data["가맹점명"].apply(clean_name)
card_data["주소_정규화2"] = card_data["소재지도로명주소"].apply(clean_address)
restaurant_data["이름_정규화"] = restaurant_data["업소명"].apply(clean_name)
restaurant_data["주소_정규화2"] = restaurant_data["소재지(도로명)"].apply(clean_address)

print("[정규화 확인]")
display(card_data[["가맹점명", "이름_정규화", "주소_정규화2"]].head(5))

[정규화 확인]


,가맹점명,이름_정규화,주소_정규화2
0,(뉴)진수성찬,(뉴)진수성찬,강원도 춘천시 공지로 450 1층
1,(사) 담배인삼공제회 (세인트콕스),(사)담배인삼공제회(세인트콕스),"강원도 춘천시 스포츠타운길399번길 22, 1층 (삼천동)"
2,(신)옹달샘,(신)옹달샘,강원도 춘천시 영서로 2352-44 상가12동83호
3,(유)아웃백스테이크하우스 춘천점,(유)아웃백스테이크하우스춘천점,강원도 춘천시 후석로 120 2층(석사동)
4,(주) 아워홈 더플레이어스GC점,(주)아워홈더플레이어스GC점,강원도 춘천시 동산면 새술막길 438 1층


## 4. 가맹점과 요식업소 1·2차 매칭

- **1차:** 양쪽에서 이름과 전체 주소가 유일하고 두 값이 정확히 일치하는 경우
- **2차:** 양쪽에서 이름이 각각 유일하며 도로명과 건물번호가 일치하는 경우

2차는 층수·호수 차이가 있을 수 있으므로 `검증대기`로 표시한다. 같은 가맹점 또는 같은 요식업소를 둘 이상에 연결하지 않도록 검사한다.

In [ ]:
keys = ["이름_정규화", "주소_정규화2"]
match_cols = ["card_id", "restaurant_id", "매칭방법", "매칭상태"]

# 1차: 상호명 + 전체 주소 일치
card_unique = card_data[
    (card_data["주소_정규화2"] != "") &
    ~card_data.duplicated(subset=keys, keep=False)
].copy()
restaurant_unique = restaurant_data[
    (restaurant_data["주소_정규화2"] != "") &
    ~restaurant_data.duplicated(subset=keys, keep=False)
].copy()

first = card_unique.merge(
    restaurant_unique, on=keys, how="inner",
    suffixes=("_card", "_restaurant"), validate="one_to_one"
)
first["매칭방법"] = "전체주소일치"
first["매칭상태"] = "1차매칭"

# 2차: 양쪽에서 상호명이 유일한 경우만 도로명+건물번호 비교
card_name_unique = card_data[
    ~card_data.duplicated(subset=["이름_정규화"], keep=False)
]
restaurant_name_unique = restaurant_data[
    ~restaurant_data.duplicated(subset=["이름_정규화"], keep=False)
]
second = card_name_unique.merge(
    restaurant_name_unique, on="이름_정규화", how="inner",
    suffixes=("_card", "_restaurant"), validate="one_to_one"
)
second = second[~second["card_id"].isin(first["card_id"])].copy()
second["도로명_card"] = second["주소_정규화2_card"].apply(get_road_address)
second["도로명_restaurant"] = second["주소_정규화2_restaurant"].apply(get_road_address)
second = second[
    (second["도로명_card"] != "") &
    (second["도로명_card"] == second["도로명_restaurant"])
].copy()
second["매칭방법"] = "기본주소일치"
second["매칭상태"] = "검증대기"

matches = pd.concat([first[match_cols], second[match_cols]], ignore_index=True)
assert matches["card_id"].is_unique
assert matches["restaurant_id"].is_unique

print("1차 전체주소 일치:", len(first))
print("2차 도로명+건물번호 일치 후보:", len(second))
print("1·2차 매칭 후보 합계:", len(matches))

1차 전체주소 일치: 622
2차 도로명+건물번호 일치 후보: 2209
1·2차 매칭 후보 합계: 2831


## 5. 미연결 원인 분석과 추가 후보 탐색

1·2차에서 매칭되지 않은 가맹점 중 **공식 데이터에 같은 이름이 한 번만 등장하는 경우**를 추려 주소를 재검토한다.

- 도로명주소가 일치하는데 가맹점 상호명이 중복된 경우: 다른 가맹점과 충돌하지 않는 단독 후보만 선택
- 도로명주소를 추출할 수 없는 경우: 동·리와 지번을 비교해 후보 추출
- 주소가 다른 경우: 자동 연결하지 않음

**주의:** 지번이 같아도 복수 점포가 있을 수 있으므로 추가로 찾은 행은 전부 `검증대기` 처리한다.

In [ ]:
# 기존 1·2차 매칭에서 미연결된 가맹점
unmatched = card_data[~card_data["card_id"].isin(matches["card_id"])].copy()

name_counts = restaurant_data["이름_정규화"].value_counts()
unmatched["동일상호_업소수"] = unmatched["이름_정규화"].map(name_counts).fillna(0).astype(int)

print("[미연결 상호명 분포]")
print("동일 상호명 없음:", (unmatched["동일상호_업소수"] == 0).sum())
print("동일 상호명 1개:", (unmatched["동일상호_업소수"] == 1).sum())
print("동일 상호명 2개 이상:", (unmatched["동일상호_업소수"] >= 2).sum())

# 이름이 유일한 요식업소를 주소 비교 대상으로 사용
restaurant_one = restaurant_data[
    ~restaurant_data.duplicated(subset=["이름_정규화"], keep=False)
][["restaurant_id", "이름_정규화", "주소_정규화2", "소재지(지번)"]]

target = unmatched[unmatched["동일상호_업소수"] == 1].copy()
check = target.merge(
    restaurant_one, on="이름_정규화", how="left",
    suffixes=("_card", "_restaurant"), validate="many_to_one"
)
check["도로명_card"] = check["주소_정규화2_card"].apply(get_road_address)
check["도로명_restaurant"] = check["주소_정규화2_restaurant"].apply(get_road_address)


def compare_road(row):
    a, b = row["도로명_card"], row["도로명_restaurant"]
    if not a or not b:
        return "도로명주소_추출불가"
    return "기본주소_일치" if a == b else "기본주소_불일치"

check["주소비교결과"] = check.apply(compare_road, axis=1)
print("\n[미연결 가맹점의 도로명주소 비교]")
print(check["주소비교결과"].value_counts())

[미연결 상호명 분포]
동일 상호명 없음: 1083
동일 상호명 1개: 535
동일 상호명 2개 이상: 119

[미연결 가맹점의 도로명주소 비교]
주소비교결과
기본주소_불일치      210
기본주소_일치       183
도로명주소_추출불가    142
Name: count, dtype: int64


In [ ]:
# 5-1. 상호명 중복으로 제외됐던 기본주소 일치 후보
same_road = check[check["주소비교결과"] == "기본주소_일치"].copy()
same_road["이름주소_중복수"] = same_road.groupby(
    ["이름_정규화", "도로명_card"]
)["card_id"].transform("count")

road_extra = same_road[same_road["이름주소_중복수"] == 1][[
    "card_id", "restaurant_id", "이름_정규화"
]].copy()
road_extra["추가매칭방법"] = "기본주소_추가검토"

# 전체 가맹점에서 동일 이름+도로명키가 유일한지 재확인
card_road = card_data[["card_id", "이름_정규화", "주소_정규화2"]].copy()
card_road["도로명키"] = card_road["주소_정규화2"].apply(get_road_address)
card_road["전체_이름주소수"] = card_road.groupby(
    ["이름_정규화", "도로명키"]
)["card_id"].transform("count")
road_extra = road_extra.merge(
    card_road[["card_id", "전체_이름주소수"]], on="card_id", how="left"
)
road_extra = road_extra[road_extra["전체_이름주소수"] == 1][[
    "card_id", "restaurant_id", "추가매칭방법"
]]

# 5-2. 도로명주소를 추출할 수 없었던 후보의 지번 비교
jibun_check = check[check["주소비교결과"] == "도로명주소_추출불가"].copy()
jibun_check["가맹점_지번"] = jibun_check["소재지도로명주소"].apply(get_jibun)
jibun_check["요식업소_지번"] = jibun_check["소재지(지번)"].apply(get_jibun)

valid = (jibun_check["가맹점_지번"] != "") & (jibun_check["요식업소_지번"] != "")
same = valid & (jibun_check["가맹점_지번"] == jibun_check["요식업소_지번"])

jibun_check["지번비교결과"] = "확인불가"
jibun_check.loc[valid, "지번비교결과"] = "불일치"
jibun_check.loc[same, "지번비교결과"] = "일치후보"

jibun_extra = jibun_check[jibun_check["지번비교결과"] == "일치후보"][ [
    "card_id", "restaurant_id"
]].copy()
jibun_extra["추가매칭방법"] = "지번주소일치"

print("[지번주소 비교 결과]")
print(jibun_check["지번비교결과"].value_counts())

# 5-3. 중복 연결 및 충돌 후보 제외
extra = pd.concat([road_extra, jibun_extra], ignore_index=True)
extra = extra[
    ~extra["card_id"].isin(matches["card_id"]) &
    ~extra["restaurant_id"].isin(matches["restaurant_id"])
].copy()
extra = extra[~extra["restaurant_id"].duplicated(keep=False)].copy()

assert extra["card_id"].is_unique
assert extra["restaurant_id"].is_unique

print("\n[추가 후보]")
print("도로명 기본주소 단독 후보:", len(road_extra))
print("지번주소 일치 후보:", len(jibun_extra))
print("중복·충돌 제외 후:", len(extra))
print(extra["추가매칭방법"].value_counts())

[지번주소 비교 결과]
지번비교결과
불일치     63
일치후보    48
확인불가    31
Name: count, dtype: int64

[추가 후보]
도로명 기본주소 단독 후보: 8
지번주소 일치 후보: 48
중복·충돌 제외 후: 48
추가매칭방법
지번주소일치       40
기본주소_추가검토     8
Name: count, dtype: int64


## 6. 최종 매칭 검증 및 주점 관련 업태 분류

기존 매칭과 추가 후보를 통합해 **가맹점 1행당 최대 1개 공식 요식업소**를 연결한다. 공식 업태가 주점 관련 분류인 경우에는 검토 그룹을 부여한다.

- `주점업태_우선점검`: 정종/대포집/소주방, 라이브카페, 감성주점
- `호프통닭_추가분류`: 호프/통닭 (치킨집 포함)

`매칭상태`와 `검토그룹`은 서로 다른 정보다. **우선점검 그룹도 실제 부적합 판정은 아니다.**

In [ ]:
# 추가 후보의 컬럼을 기존 매칭 결과와 통일
extra_matches = extra.rename(columns={"추가매칭방법": "매칭방법"}).copy()
extra_matches["매칭상태"] = "검증대기"
matches_v2 = pd.concat(
    [matches[match_cols], extra_matches[match_cols]], ignore_index=True
)

assert matches_v2["card_id"].is_unique
assert matches_v2["restaurant_id"].is_unique

# 원래 가맹점 전체에 공식 업태 추가
restaurant_info = restaurant_data[[
    "restaurant_id", "업소명", "업종명", "업태명", "소재지(도로명)"
]].rename(columns={
    "업종명": "공식_업종명",
    "업태명": "공식_업태명",
    "소재지(도로명)": "요식업소_주소"
})

final_v2 = card_data.merge(
    matches_v2, on="card_id", how="left", validate="one_to_one"
)
# 미연결 행은 restaurant_id가 여러 개 NaN이므로 many_to_one 검증
final_v2 = final_v2.merge(
    restaurant_info, on="restaurant_id", how="left", validate="many_to_one"
)
final_v2["매칭상태"] = final_v2["매칭상태"].fillna("미연결")
final_v2["매칭방법"] = final_v2["매칭방법"].fillna("없음")

assert len(final_v2) == len(card_data)
assert final_v2["card_id"].is_unique
assert final_v2["restaurant_id"].notna().sum() == len(matches_v2)

bar_types = ["정종/대포집/소주방", "라이브카페", "감성주점", "호프/통닭"]
bar_final = final_v2[final_v2["공식_업태명"].isin(bar_types)].copy()


def classify_type(value):
    if value in ["정종/대포집/소주방", "라이브카페", "감성주점"]:
        return "주점업태_우선점검"
    if value == "호프/통닭":
        return "호프통닭_추가분류"
    return "기타"

bar_final["검토그룹"] = bar_final["공식_업태명"].apply(classify_type)
bar_final["매칭검증결과"] = ""
bar_final["실제영업형태"] = ""
bar_final["비고"] = ""

print("[최종 데이터 현황]")
print("가맹점 전체:", len(final_v2))
print("매칭 및 후보:", len(matches_v2))
print("미연결:", (final_v2["매칭상태"] == "미연결").sum())
print("매칭 후보 비율:", round(len(matches_v2) / len(final_v2) * 100, 2), "%")
print("\n[매칭 방법]")
print(final_v2["매칭방법"].value_counts())
print("\n[주점 관련 공식 업태 후보]")
print(pd.crosstab(bar_final["공식_업태명"], bar_final["매칭상태"]))

# 원본 CSV가 동일한 버전이라면 아래 숫자와 비교
expected = {"가맹점": 4568, "매칭후보": 2879, "주점관련후보": 197}
actual = {"가맹점": len(final_v2), "매칭후보": len(matches_v2), "주점관련후보": len(bar_final)}
print("\n[이전 실행 결과와 비교]")
for key in expected:
    print(f"{key}: {actual[key]} (이전 {expected[key]})", "일치" if actual[key] == expected[key] else "차이 있음: 입력 데이터 및 매칭 조건 확인")

[최종 데이터 현황]
가맹점 전체: 4568
매칭 및 후보: 2879
미연결: 1689
매칭 후보 비율: 63.03 %

[매칭 방법]
매칭방법
기본주소일치       2209
없음           1689
전체주소일치        622
지번주소일치         40
기본주소_추가검토       8
Name: count, dtype: int64

[주점 관련 공식 업태 후보]
매칭상태        1차매칭  검증대기
공식_업태명                
라이브카페          0     2
정종/대포집/소주방     7    40
호프/통닭         28   120

[이전 실행 결과와 비교]
가맹점: 4568 (이전 4568) 일치
매칭후보: 2879 (이전 2879) 일치
주점관련후보: 197 (이전 197) 일치


## 7. 최종 CSV 저장

최종 데이터는 Google Drive의 `data/processed` 폴더에 보관한다.

| 파일 | 용도 |
|---|---|
| `가맹점_매칭결과_최종후보.csv` | 전체 가맹점 4,568행 기준 결과 |
| `검증대기_가맹점_최종후보.csv` | 수동으로 연결 여부를 확인할 후보 |
| `미연결_가맹점_최종후보.csv` | 현재 공식 요식업소를 연결하지 못한 가맹점 |
| `주점업태_점검후보_최종후보.csv` | 주점 관련 업태 점검 후보 (호프/통닭 구분 포함) |

**팀원에게 공유할 때:** 이 파일들은 *전처리 및 자동 매칭 후보 결과*이며, 수동 검증을 완료한 확정된 적합성 판단 데이터는 아니다. 분석 시 `매칭상태`를 필수로 확인해야 한다.

In [ ]:
output_files = {
    "가맹점_매칭결과_최종후보.csv": final_v2,
    "검증대기_가맹점_최종후보.csv": final_v2[final_v2["매칭상태"] == "검증대기"],
    "미연결_가맹점_최종후보.csv": final_v2[final_v2["매칭상태"] == "미연결"],
    "주점업태_점검후보_최종후보.csv": bar_final,
}

for name, data in output_files.items():
    path = os.path.join(OUTPUT_DIR, name)
    data.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {name} ({len(data):,}행)")

print("\n저장 위치:", OUTPUT_DIR)
print("최종 전처리 파일 수:", len(output_files))

저장 완료: 가맹점_매칭결과_최종후보.csv (4,568행)
저장 완료: 검증대기_가맹점_최종후보.csv (2,257행)
저장 완료: 미연결_가맹점_최종후보.csv (1,689행)
저장 완료: 주점업태_점검후보_최종후보.csv (197행)

저장 위치: /content/drive/MyDrive/data/processed
최종 전처리 파일 수: 4


## 마무리 및 후속 검증

### 전처리에서 수행한 작업
1. 가맹점과 요식업소의 완전 중복 행을 제거했다.
2. 상호명·주소 표기를 비교 가능한 형태로 정규화했다.
3. 전체주소, 도로명+건물번호, 지번주소를 활용해 중복 없이 매칭 후보를 구성했다.
4. 공식 업태를 연결하고 주점 관련 업태 후보를 별도로 추출했다.

### 아직 남아 있는 한계
- 주소·상호명 일치만으로 실제 동일 업소를 보증하지 못한다.
- 동일 이름으로 여러 점포가 존재하거나 상세 호수가 달라 연결이 누락될 수 있다.
- 공식 업태가 최신 영업 형태와 다를 수 있다.
- **요식업소와 연결되지 않았다는 이유만으로 급식카드 가맹점이 부적절한 것은 아니다.** 편의점·제과점 등은 데이터 범위가 다를 수 있다.
- 현장 또는 지도·메뉴 확인 없이 주점 관련 업태 후보를 부적합 업소로 확정해서는 안 된다.

춘천시 아동급식카드 가맹점과 요식업소 현황을 이름 및 주소 기준으로 정규화·결합했습니다. 완전 중복 제거 후 가맹점 4,568행을 분석했으며, 공식 업태가 연결된 매칭 및 검증 후보는 2,879행(63.03%)입니다. 이 중 전체주소 일치는 622행, 나머지 2,257행은 추가 검증이 필요한 자동 매칭 후보입니다. 주점 관련 공식 업태 후보는 197행으로, 호프/통닭 148행과 기타 주점 관련 업태 49행을 구분했습니다. 이는 최종 부적합 업소 수가 아니며, 매칭과 실제 영업 형태의 확인이 필요합니다.